# 熊本日日新聞の記事検索結果を取得する

熊本日日新聞で「熊本地震」を検索し、検索欄の合計件数までタイトル・本文・公開日時・URLを取得します。

- 検索結果の「さらに表示」に対応するページを、表示された合計件数まで取得します。
- 熊日電子版へログインし、契約上閲覧できる本文だけを取得します。アクセス制限の回避は行いません。
- 短時間に大量アクセスしないよう、記事ごとに待機時間を設けています。
- 実行前に、利用規約・著作権・robots.txtと契約内容を確認してください。
- サイトの画面構成が変わった場合は、セレクタの調整が必要になることがあります。

In [1]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests", "beautifulsoup4", "pandas", "playwright"
])


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import json
import math
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://kumanichi.com"
LOGIN_URL = f"{BASE_URL}/user/login"
SEARCH_URL = f"{BASE_URL}/search/cse"
KEYWORD = "熊本地震"
SEARCH_PAGE_INTERVAL = 1.0
ARTICLE_INTERVAL = 1.5
CONNECT_TIMEOUT = 15
READ_TIMEOUT = 90
MAX_RETRIES = 4
MAX_SEARCH_PAGES = 1000  # 異常時の無限ループ防止

session = requests.Session()
retry = Retry(
    total=MAX_RETRIES,
    connect=MAX_RETRIES,
    read=MAX_RETRIES,
    status=MAX_RETRIES,
    backoff_factor=1.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET", "HEAD"}),
    respect_retry_after_header=True,
    raise_on_status=False,
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})

## ログイン情報の入力

IDとパスワードは実行中の変数にだけ保持され、ノートブックファイルには保存されません。パスワードは入力中も画面に表示されません。

In [3]:
from getpass import getpass

KUMANICHI_ID = input("熊日電子版のログインID（メールアドレス）: ").strip()
KUMANICHI_PASSWORD = getpass("パスワード: ")

In [4]:
from playwright.async_api import async_playwright


async def _first_visible(page, selectors):
    for selector in selectors:
        locator = page.locator(selector)
        for index in range(await locator.count()):
            candidate = locator.nth(index)
            if await candidate.is_visible():
                return candidate
    return None


async def login_to_kumanichi(requests_session, login_id, password):
    """Chromeで熊日電子版へログインし、Cookieをrequestsへ渡す。"""
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(channel="chrome", headless=False)
    context = await browser.new_context(locale="ja-JP")
    page = await context.new_page()

    try:
        await page.goto(
            f"{LOGIN_URL}?destination=/search/cse",
            wait_until="domcontentloaded", timeout=60000,
        )

        # 最初の画面は外部認証サービスへ進むためのボタンだけを表示する。
        auth_start = page.locator('input[name="aws_cognito"][type="submit"]')
        if await auth_start.count() == 1:
            await auth_start.click()
            await page.wait_for_load_state("domcontentloaded")

        email = await _first_visible(page, [
            'input[type="email"]', 'input[name="username"]',
            'input[name*="email" i]', 'input[name*="user" i]',
        ])
        password_input = await _first_visible(page, ['input[type="password"]'])
        if email is not None and password_input is not None:
            await email.fill(login_id)
            await password_input.fill(password)
            submit = await _first_visible(page, [
                'button[type="submit"]', 'input[type="submit"]',
            ])
            if submit is not None:
                await submit.click()
                await page.wait_for_timeout(1500)

        # MFA・追加確認・認証画面の変更があっても手動で完了できるようにする。
        if "kumanichi.com" not in page.url or "/user/login" in page.url:
            input(
                "Chromeでログイン（必要なら追加認証）を完了してから、"
                "ここで Enter を押してください: "
            )

        await page.goto(BASE_URL, wait_until="domcontentloaded", timeout=60000)
        page_text = await page.locator("body").inner_text()
        if "ログアウト" not in page_text and "マイページ" not in page_text:
            print("注意: ログイン表示を確認できませんでした。会員記事で本文取得可否を判定します。")

        for cookie in await context.cookies():
            if "kumanichi" in cookie.get("domain", ""):
                requests_session.cookies.set(
                    cookie["name"], cookie["value"],
                    domain=cookie.get("domain"), path=cookie.get("path", "/"),
                )
    finally:
        await context.close()
        await browser.close()
        await playwright.stop()


await login_to_kumanichi(session, KUMANICHI_ID, KUMANICHI_PASSWORD)
print("ログインCookieを取得しました。")

注意: ログイン表示を確認できませんでした。会員記事で本文取得可否を判定します。
ログインCookieを取得しました。


In [5]:
def get_soup(url, *, params=None):
    response = session.get(
        url, params=params, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT)
    )
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")


def _displayed_total(soup):
    node = soup.select_one(".site-search-results")
    if not node:
        return None
    match = re.search(r"([0-9０-９,，]+)\s*件", node.get_text(" ", strip=True))
    if not match:
        return None
    number = match.group(1).translate(str.maketrans("０１２３４５６７８９，", "0123456789,"))
    return int(number.replace(",", ""))


def _search_result_urls(soup):
    urls = []
    seen = set()
    for link in soup.select('.view-content .y2024-news-list__item > a[href^="/articles/"]'):
        url = urljoin(BASE_URL, link.get("href", ""))
        if re.fullmatch(r"https://kumanichi\.com/articles/\d+", url) and url not in seen:
            seen.add(url)
            urls.append(url)
    return urls


def collect_search_urls(keyword):
    """表示合計件数まで「さらに表示」相当の結果を取得する。"""
    first_soup = get_soup(SEARCH_URL, params={"search_api_fulltext": keyword})
    total_count = _displayed_total(first_soup)
    first_urls = _search_result_urls(first_soup)
    if total_count is None:
        raise RuntimeError("検索画面に表示された合計件数を読み取れませんでした。")
    if total_count == 0:
        return [], 0
    if not first_urls:
        raise RuntimeError("検索結果の記事URLを取得できませんでした。")

    page_size = len(first_urls)
    last_page_index = math.ceil(total_count / page_size) - 1
    if last_page_index >= MAX_SEARCH_PAGES:
        raise RuntimeError(
            f"必要ページ数が安全上限 {MAX_SEARCH_PAGES} を超えました。"
        )

    print(f"検索結果: {total_count:,}件 / 初期表示 {page_size}件")
    if last_page_index == 0:
        urls = first_urls
    else:
        # このサイトのpage指定は先頭から指定ページまでを累積表示する。
        last_soup = get_soup(
            SEARCH_URL,
            params={"search_api_fulltext": keyword, "page": last_page_index},
        )
        urls = _search_result_urls(last_soup)
        print(f"最終ページ相当を取得: {len(urls):,}/{total_count:,}件")

    # DOMやページング仕様が変わった場合は、次リンクを順番に追う方式へ切り替える。
    if len(urls) != total_count:
        print("累積取得の件数が合わないため、ページを順番に確認します。")
        urls = []
        seen = set()
        page_index = 0
        while page_index < MAX_SEARCH_PAGES and len(urls) < total_count:
            soup = get_soup(
                SEARCH_URL,
                params={"search_api_fulltext": keyword, "page": page_index},
            )
            before = len(urls)
            for url in _search_result_urls(soup):
                if url not in seen:
                    seen.add(url)
                    urls.append(url)
            print(f"検索 {page_index + 1}ページ目: {len(urls):,}/{total_count:,}件")
            if len(urls) <= before:
                break
            if not soup.select_one('a[rel="next"][href]'):
                break
            page_index += 1
            time.sleep(SEARCH_PAGE_INTERVAL)

    if len(urls) != total_count:
        raise RuntimeError(
            f"検索欄は{total_count:,}件ですが、重複除去後に{len(urls):,}件でした。"
        )
    return urls, total_count


def _published_at(soup):
    settings = soup.select_one('script[data-drupal-selector="drupal-settings-json"]')
    if settings:
        try:
            data = json.loads(settings.string or settings.get_text())
            value = data.get("ga_variable", {}).get("published_date", "")
            if value:
                return value
        except (json.JSONDecodeError, TypeError):
            pass
    date_area = soup.select_one(".y2024-heading-wrap-02 .y2024-text-date")
    if date_area:
        match = re.search(
            r"(\d{4})年(\d{1,2})月(\d{1,2})日\s+(\d{1,2}):(\d{2})",
            date_area.get_text(" ", strip=True),
        )
        if match:
            year, month, day, hour, minute = map(int, match.groups())
            return f"{year:04d}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}"
    return ""


def fetch_article(url):
    soup = get_soup(url)
    heading = soup.select_one(".y2024-heading-wrap-02 h1, main h1, h1")
    title = heading.get_text(" ", strip=True) if heading else ""
    published_at = _published_at(soup)

    body_node = soup.select_one(".y2024-article__text.field--name-field-rich")
    body_parts = []
    if body_node:
        for node in body_node.find_all(["p", "h2", "h3"], recursive=False):
            text = node.get_text(" ", strip=True)
            if text and text not in body_parts:
                body_parts.append(text)
    body = "\n".join(body_parts)

    paywall = soup.select_one(".box_card__title, #login_register")
    body_is_excerpt = paywall is not None and not body
    if body_is_excerpt:
        extraction_error = "契約またはログイン状態により本文が表示されていません"
    elif not body:
        extraction_error = "本文を取得できませんでした"
    else:
        extraction_error = ""

    return {
        "title": title,
        "body": body,
        "published_at": published_at,
        "url": url,
        "body_is_excerpt": body_is_excerpt,
        "extraction_error": extraction_error,
    }


def collect_articles(urls):
    records = []
    for index, url in enumerate(urls, start=1):
        try:
            record = fetch_article(url)
        except Exception as exc:
            record = {
                "title": "", "body": "", "published_at": "", "url": url,
                "body_is_excerpt": False,
                "extraction_error": f"{type(exc).__name__}: {exc}",
            }
        records.append(record)
        print(f"本文 {index:,}/{len(urls):,}: {record['title'] or url}")
        if index < len(urls):
            time.sleep(ARTICLE_INTERVAL)
    return records

In [6]:
search_urls, displayed_total = collect_search_urls(KEYWORD)
articles = collect_articles(search_urls)

df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

assert len(df) == displayed_total, (
    f"検索欄の合計 {displayed_total:,}件に対し、データは {len(df):,}件です。"
)
df

検索結果: 7,428件 / 初期表示 10件
最終ページ相当を取得: 0/7,428件
累積取得の件数が合わないため、ページを順番に確認します。
検索 1ページ目: 10/7,428件
検索 2ページ目: 20/7,428件
検索 3ページ目: 30/7,428件
検索 4ページ目: 40/7,428件
検索 5ページ目: 50/7,428件
検索 6ページ目: 60/7,428件
検索 7ページ目: 70/7,428件
検索 8ページ目: 80/7,428件
検索 9ページ目: 90/7,428件
検索 10ページ目: 100/7,428件
検索 11ページ目: 110/7,428件
検索 12ページ目: 120/7,428件
検索 13ページ目: 130/7,428件
検索 14ページ目: 140/7,428件
検索 15ページ目: 150/7,428件
検索 16ページ目: 160/7,428件
検索 17ページ目: 170/7,428件
検索 18ページ目: 180/7,428件
検索 19ページ目: 190/7,428件
検索 20ページ目: 200/7,428件
検索 21ページ目: 209/7,428件
検索 22ページ目: 219/7,428件
検索 23ページ目: 229/7,428件
検索 24ページ目: 239/7,428件
検索 25ページ目: 249/7,428件
検索 26ページ目: 259/7,428件
検索 27ページ目: 269/7,428件
検索 28ページ目: 279/7,428件
検索 29ページ目: 289/7,428件
検索 30ページ目: 299/7,428件
検索 31ページ目: 309/7,428件
検索 32ページ目: 319/7,428件
検索 33ページ目: 328/7,428件
検索 34ページ目: 338/7,428件
検索 35ページ目: 348/7,428件
検索 36ページ目: 357/7,428件
検索 37ページ目: 367/7,428件
検索 38ページ目: 377/7,428件
検索 39ページ目: 387/7,428件
検索 40ページ目: 397/7,428件
検索 41ページ目: 407/7,428件
検索 42ページ目: 417/7,428件
検索 43ページ目: 427/7,428

ConnectionError: HTTPSConnectionPool(host='kumanichi.com', port=443): Max retries exceeded with url: /search/cse?search_api_fulltext=%E7%86%8A%E6%9C%AC%E5%9C%B0%E9%9C%87&page=224 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='kumanichi.com', port=443): Read timed out. (read timeout=90)"))

In [ ]:
output_path = Path("kumanichi_熊本地震.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")